In [1]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.8 MB/s eta 0:00:00


In [2]:
import torch

print("CUDA Available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("Still running on CPU!")

CUDA Available?: True
Device Name: Tesla T4


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [6]:
base_model_id = "Qwen/Qwen2.5-1.5B"

In [9]:
#loading the tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"

In [22]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [23]:
#prepare model for 4bit QLORA
model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

In [24]:
# configuring the lora adapters
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [14]:
#loading the dataset
dataset = load_dataset("gbharti/finance-alpaca", split="train[:500]")

README.md:   0%|          | 0.00/831 [00:00<?, ?B/s]

Cleaned_date.json: reconstructing file:   0%|          |  0.00B / 42.9MB            

Cleaned_date.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/68912 [00:00<?, ? examples/s]

In [15]:
#formating the dataset rows into instruction response template
def apply_chat_template(sample):
    sample["text"] = f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['output']}"
    return sample

sft_dataset = dataset.map(apply_chat_template)

print("Pre-formatted Text Sample:")
print(sft_dataset[0]["text"])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Pre-formatted Text Sample:
### Instruction:
For a car, what scams can be plotted with 0% financing vs rebate?

### Response:
The car deal makes money 3 ways. If you pay in one lump payment. If the payment is greater than what they paid for the car, plus their expenses, they make a profit. They loan you the money. You make payments over months or years, if the total amount you pay is greater than what they paid for the car, plus their expenses, plus their finance expenses they make money. Of course the money takes years to come in, or they sell your loan to another business to get the money faster but in a smaller amount. You trade in a car and they sell it at a profit. Of course that new transaction could be a lump sum or a loan on the used car... They or course make money if you bring the car back for maintenance, or you buy lots of expensive dealer options. Some dealers wave two deals in front of you: get a 0% interest loan. These tend to be shorter 12 months vs 36,48,60 or even 72 m

In [29]:
#supervised fine tuning conig
use_cuda = torch.cuda.is_available()

sft_args = SFTConfig(
    output_dir="./sft_financial_model",
    dataset_text_field="text",
    max_length=256,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=40,
    fp16=False,                 # <--- Turn this OFF to disable GradScaler
    bf16=True,                  # <--- Turn this ON to bypass the bug
    gradient_checkpointing=False,
    logging_steps=1,
    logging_first_step=True
)

In [30]:
#making the trainer object
sft_trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    peft_config=peft_config,
    args=sft_args,
    processing_class=tokenizer
)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [31]:
#trained and save the trained Lora Adapter
sft_trainer.train()

sft_trainer.model.save_pretrained("./sft_financial_adapter")
tokenizer.save_pretrained("./sft_financial_adapter")
print("Training Complete and model is saved")

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,2.508177
2,2.710464
3,3.010367
4,2.869235
5,2.610268
6,2.878485
7,2.786004
8,2.795634
9,2.608721
10,2.756510


Training Complete and model is saved


In [32]:
# lets check some input and results of model after QLORA
from peft import PeftModel

adapter_path = "./sft_financial_adapter"
base_model_id = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# Define your test financial prompt
test_prompt = "### Instruction:\nWhat is the difference between a stock and a bond?\n\n### Response:\n"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

print("Generating response from your fine-tuned model...\n")
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and print the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generating response from your fine-tuned model...

### Instruction:
What is the difference between a stock and a bond?

### Response:
You could also call it a "stock option" (a share of ownership in a company).   In a bond you own a piece of the company.  In a stock you own a share of the company.  So you get more of a "piece" of the company if the company does well.  You get more of a "piece" of the company if the company does badly.  And you can sell your share at any time.  So you are more of a "stock" than a bond.


# RHLF

In [70]:
from trl.experimental.ppo import PPOTrainer, PPOConfig
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification
import torch
import torch.nn as nn

In [73]:
ref_model = None

In [105]:
from transformers import AutoModelForSequenceClassification
from peft import prepare_model_for_kbit_training

# 1. Load the Actor
actor_model = AutoModelForCausalLM.from_pretrained(
    "./sft_financial_adapter",
    quantization_config=bnb_config,
    device_map="auto"
)
actor_model = prepare_model_for_kbit_training(actor_model)
actor_model.gradient_checkpointing_enable() # <-- Force Memory Saving

# 2. Load the Value Model
value_model = AutoModelForSequenceClassification.from_pretrained(
    "./sft_financial_adapter",
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto"
)
value_model = prepare_model_for_kbit_training(value_model)
value_model.gradient_checkpointing_enable() # <-- Force Memory Saving

# 3. REWARD MODEL MEMORY FIX: Point it to reuse value_model's weights
# This stops PyTorch from creating a 3rd Qwen instance in VRAM
reward_model = value_model

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 892.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 325.81 MiB is free. Including non-PyTorch memory, this process has 14.24 GiB memory in use. Of the allocated memory 13.67 GiB is allocated by PyTorch, and 443.28 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [92]:
# 5. Tokenize dataset for PPO and drop all raw string features
def tokenize_ppo_data(sample):
    formatted = f"### Instruction:\n{sample['instruction']}\n\n### Response:\n"
    tokenized = tokenizer(formatted, truncation=True) # Automatically creates input_ids and attention_mask
    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"]
    }

# We dynamically fetch all original text column names and strip them away
ppo_dataset = sft_dataset.map(
    tokenize_ppo_data,
    remove_columns=sft_dataset.column_names # <-- THIS WIPE CLEARS THE VALUEERROR
)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [93]:
ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=2,
    mini_batch_size=1,
    num_ppo_epochs=1
)

In [101]:
ppo_trainer = PPOTrainer(
    args=ppo_config,
    processing_class=tokenizer,
    model=actor_model,
    ref_model=ref_model,
    reward_model=reward_model,
    value_model=value_model,
    train_dataset=ppo_dataset
)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.41 GiB is allocated by PyTorch, and 6.19 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [98]:
ppo_trainer.train()

print("PPO RLHF Training Completed Successfully!")

# Save your final fully-aligned policy model
ppo_trainer.model.save_pretrained("./final_financial_ppo_model")
tokenizer.save_pretrained("./final_financial_ppo_model")
print("Final PPO model saved")

===training policy===


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [102]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()
print("GPU memory cache cleared!")

GPU memory cache cleared!


In [103]:
actor_model.gradient_checkpointing_enable()
value_model.gradient_checkpointing_enable()


In [104]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
